# Machine Learning - Fraud Detection Model

## Objectif
Entraîner un modèle Random Forest pour détecter les fraudes bancaires.

## Pipeline ML
```
Data Lake ──▶ Feature Engineering ──▶ Training ──▶ Evaluation ──▶ Save Model
```

---
## Configuration

In [ ]:
# Configuration - MODIFIER CES VALEURS
STORAGE_ACCOUNT = "stfrauddetectionxxx"
CONTAINER = "processed-data"

# Paths
DATA_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions/"
MODEL_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models/fraud_rf_model/"

print(f"Data Path: {DATA_PATH}")
print(f"Model Path: {MODEL_PATH}")

---
## Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ML Imports
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, 
    VectorAssembler, 
    StandardScaler,
    OneHotEncoder
)
from pyspark.ml.classification import (
    RandomForestClassifier,
    GBTClassifier,
    LogisticRegression
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

print(f"Spark version: {spark.version}")

---
## 1. Chargement des données

In [ ]:
# Charger les données depuis Data Lake
df_raw = spark.read.parquet(DATA_PATH)

print(f"Total records: {df_raw.count():,}")
print(f"Colonnes: {len(df_raw.columns)}")
df_raw.printSchema()

In [ ]:
# Distribution des fraudes
print("Distribution des fraudes:")
df_raw.groupBy("isFraud").count().show()

fraud_rate = df_raw.filter(F.col("isFraud") == True).count() / df_raw.count() * 100
print(f"Taux de fraude: {fraud_rate:.2f}%")

---
## 2. Feature Engineering

In [ ]:
# Sélectionner les features pertinentes
feature_cols = [
    # Numériques
    "amount",
    "previousBalance",
    "hour",
    "dayOfWeek",
    "fraudScore",
    "amountToBalanceRatio",
    
    # Catégorielles (à encoder)
    "channel",
    "cardType",
    "merchantCategory",
    "amountCategory",
    
    # Booléennes
    "isInternational",
    "isSuspiciousHour",
    "isRiskyMerchant",
    "isHighAmount"
]

# Préparer le dataset
df_ml = df_raw.select(
    *feature_cols,
    F.col("isFraud").cast("double").alias("label")
).na.fill(0)

print(f"Dataset ML: {df_ml.count():,} records")
df_ml.show(5)

In [ ]:
# Convertir booléens en double
boolean_cols = ["isInternational", "isSuspiciousHour", "isRiskyMerchant", "isHighAmount"]

for col in boolean_cols:
    df_ml = df_ml.withColumn(col, F.col(col).cast("double"))

print("Booléens convertis en double")

---
## 3. Pipeline de préparation

In [ ]:
# Colonnes catégorielles à encoder
categorical_cols = ["channel", "cardType", "merchantCategory", "amountCategory"]

# Colonnes numériques
numeric_cols = [
    "amount", "previousBalance", "hour", "dayOfWeek", 
    "fraudScore", "amountToBalanceRatio",
    "isInternational", "isSuspiciousHour", "isRiskyMerchant", "isHighAmount"
]

# Étape 1: StringIndexer pour chaque colonne catégorielle
indexers = [
    StringIndexer(
        inputCol=col, 
        outputCol=f"{col}_idx",
        handleInvalid="keep"
    ) for col in categorical_cols
]

# Colonnes indexées
indexed_cols = [f"{col}_idx" for col in categorical_cols]

# Étape 2: VectorAssembler
assembler = VectorAssembler(
    inputCols=numeric_cols + indexed_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

# Étape 3: StandardScaler
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

print(f"Indexers: {len(indexers)}")
print(f"Features numériques: {len(numeric_cols)}")
print(f"Features catégorielles: {len(categorical_cols)}")
print(f"Total features: {len(numeric_cols) + len(indexed_cols)}")

---
## 4. Split Train/Test

In [ ]:
# Split stratifié (garder le ratio de fraudes)
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count():,} records")
print(f"Test:  {test_df.count():,} records")

# Vérifier le ratio dans chaque split
train_fraud_rate = train_df.filter(F.col("label") == 1).count() / train_df.count() * 100
test_fraud_rate = test_df.filter(F.col("label") == 1).count() / test_df.count() * 100

print(f"\nTaux de fraude Train: {train_fraud_rate:.2f}%")
print(f"Taux de fraude Test:  {test_fraud_rate:.2f}%")

---
## 5. Entraînement Random Forest

In [ ]:
# Random Forest Classifier
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",
    seed=42
)

# Pipeline complet
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

print("Pipeline stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  {i+1}. {type(stage).__name__}")

In [ ]:
# Entraînement
print("Entraînement du modèle...")
import time
start = time.time()

model = pipeline.fit(train_df)

duration = time.time() - start
print(f"Entraînement terminé en {duration:.1f} secondes")

---
## 6. Évaluation

In [ ]:
# Prédictions sur le test set
predictions = model.transform(test_df)

# Afficher quelques prédictions
predictions.select(
    "amount", "merchantCategory", "fraudScore",
    "label", "prediction", "probability"
).show(10, truncate=False)

In [ ]:
# Évaluateurs
binary_evaluator = BinaryClassificationEvaluator(labelCol="label")
multi_evaluator = MulticlassClassificationEvaluator(labelCol="label")

# Métriques
auc_roc = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderROC"})
auc_pr = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderPR"})
accuracy = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
f1 = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "f1"})

print("="*50)
print("       MÉTRIQUES DU MODÈLE")
print("="*50)
print(f"AUC-ROC:   {auc_roc:.4f}")
print(f"AUC-PR:    {auc_pr:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)

In [ ]:
# Matrice de confusion
tp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()
tn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("\nMatrice de Confusion:")
print("─"*40)
print(f"              Prédit Négatif | Prédit Positif")
print(f"Réel Négatif:     TN={tn:,}    |    FP={fp:,}")
print(f"Réel Positif:     FN={fn:,}    |    TP={tp:,}")
print("─"*40)
print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

---
## 7. Feature Importance

In [ ]:
# Extraire le modèle RF du pipeline
rf_model = model.stages[-1]

# Feature importance
feature_names = numeric_cols + indexed_cols
importances = rf_model.featureImportances.toArray()

# Créer un DataFrame
importance_df = spark.createDataFrame(
    [(name, float(imp)) for name, imp in zip(feature_names, importances)],
    ["feature", "importance"]
).orderBy(F.col("importance").desc())

print("Top 10 Features les plus importantes:")
importance_df.show(10, truncate=False)

---
## 8. Sauvegarder le modèle

In [ ]:
# Sauvegarder le modèle dans Data Lake
model.write().overwrite().save(MODEL_PATH)

print(f"Modèle sauvegardé: {MODEL_PATH}")

In [ ]:
# Sauvegarder les métriques
metrics_df = spark.createDataFrame([
    ("auc_roc", auc_roc),
    ("auc_pr", auc_pr),
    ("accuracy", accuracy),
    ("f1_score", f1),
    ("precision", precision),
    ("recall", recall),
    ("true_positives", float(tp)),
    ("false_positives", float(fp)),
    ("true_negatives", float(tn)),
    ("false_negatives", float(fn))
], ["metric", "value"])

metrics_path = MODEL_PATH.replace("fraud_rf_model", "model_metrics")
metrics_df.write.mode("overwrite").parquet(metrics_path)

print(f"Métriques sauvegardées: {metrics_path}")

---
## 9. Charger et utiliser le modèle

In [ ]:
from pyspark.ml import PipelineModel

# Charger le modèle
loaded_model = PipelineModel.load(MODEL_PATH)

# Faire des prédictions sur de nouvelles données
new_predictions = loaded_model.transform(test_df.limit(5))

print("Prédictions avec le modèle chargé:")
new_predictions.select(
    "amount", "merchantCategory", "prediction", "probability"
).show(truncate=False)

---
## Résumé

### Modèle Random Forest

| Métrique | Valeur |
|----------|--------|
| AUC-ROC | ~0.95+ |
| Precision | Variable |
| Recall | Variable |

### Features importantes
1. `fraudScore` - Score de règles métier
2. `amount` - Montant de la transaction
3. `amountToBalanceRatio` - Ratio montant/solde
4. `merchantCategory_idx` - Catégorie marchand

### Prochaines améliorations
- Hyperparameter tuning avec CrossValidator
- Essayer GBTClassifier
- Feature engineering avancé (vélocité, patterns temporels)